# Población mundial


En 1968, Paul Erlich publicó predijo que la población mundial crecería rápidamente durante la década de 1970, que la producción agrícola no podría mantenerse al ritmo y que la hambruna masiva en las dos décadas siguientes sería inevitable. Predicciones que fueron erróneas.

Sin embargo, el crecimiento de la población mundial sigue siendo un tema de preocupación, y sigue abierta la pregunta de cuántas personas puede sostener la Tierra mientras mantenemos y mejoramos nuestra calidad de vida.

En este notebook, utilizaremos herramientas de los notebooks anteriores para cargar datos, explorar cómo ha cambiado la población mundial desde 1950 y generar predicciones para los próximos 50 a 100 años.



Este capítulo está disponible como un cuaderno de Jupyter donde puedes leer el texto, ejecutar el código y trabajar en los ejercicios.
Haz clic aquí para acceder a los cuadernos: <https://allendowney.github.io/ModSimPy/>.


## Crecimiento de la población mundial

El artículo de Wikipedia sobre la población mundial contiene tablas con estimaciones históricas y proyecciones para el futuro (<https://en.wikipedia.org/wiki/Estimates_of_historical_world_population>).



La siguiente función permite descargar una copia de https://en.wikipedia.org/wiki/World_population_estimates


In [ ]:
def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

In [4]:
download('https://raw.githubusercontent.com/AllenDowney/' +
         'ModSimPy/master/data/World_population_estimates.html')

Para leer estos datos, usaremos la biblioteca Pandas, que proporciona funciones para trabajar con datos. La función que utilizaremos es `read_html`, que lee un documento HTML y devuelve las tablas que contiene. Antes de poder usarla, debemos importarla.


In [5]:
from pandas import read_html

Ahora podemos usarla así:


In [6]:
filename = 'World_population_estimates.html'
tables = read_html(filename,
                   header=0, 
                   index_col=0,
                   decimal='M')

Los argumentos son:

-   `filename`: El nombre del archivo (incluyendo el directorio en el que se encuentra) como una cadena. Este argumento también puede ser una URL que comience con `http`.

-   `header`: Indica qué fila de cada tabla debe considerarse la *cabecera*, es decir, el conjunto de etiquetas que identifican las columnas. En este caso es la primera fila (numerada como 0).

-   `index_col`: Indica qué columna de cada tabla debe considerarse el *índice*, es decir, el conjunto de etiquetas que identifican las filas. En este caso es la primera columna, que contiene los años.

-   `decimal`: Normalmente este argumento se usa para indicar qué carácter debe considerarse como punto decimal, porque algunas convenciones usan un punto y otras una coma. En este caso, estoy "forzando" la característica tratando `M` como punto decimal, lo que permite que algunas de las estimaciones, que están expresadas en millones, se lean como números.

El resultado, que se asigna a `tables`, es una secuencia que contiene un `DataFrame` por cada tabla. Un `DataFrame` es un objeto, definido por Pandas, que representa datos tabulares.

Para seleccionar un `DataFrame` de `tables`, podemos usar el operador de corchetes así:


In [7]:
table2 = tables[2]

Esta línea selecciona la tercera tabla (numerada como 2), que contiene estimaciones de población desde 1950 hasta 2016.

Podemos usar `head` para mostrar las primeras líneas de la tabla.


In [8]:
table2.head()

La primera columna, etiquetada `Year`, es especial. Es el índice de este `DataFrame`, lo que significa que contiene las etiquetas de las filas.

Algunos valores usan notación científica; por ejemplo, `2.516000e+09` es una abreviatura de $2.516 \cdot 10^9$.

`NaN` es un valor especial que indica datos faltantes.


Las etiquetas de las columnas son cadenas largas, lo que dificulta trabajar con ellas.
Podemos reemplazarlas por cadenas más cortas de esta manera:


In [9]:
table2.columns = ['census', 'prb', 'un', 'maddison', 
                  'hyde', 'tanton', 'biraben', 'mj', 
                  'thomlinson', 'durand', 'clark']

Ahora podemos seleccionar una columna del `DataFrame` usando el operador punto, como cuando seleccionamos una variable de estado de un objeto `Estado`.

Aquí están las estimaciones de la Oficina del Censo de los Estados Unidos:


In [10]:
census = table2.census / 1e9

El resultado es una `Series` de Pandas, que es similar a los objetos `TimeSeries` y `SweepSeries` que hemos estado usando.

El número `1e9` es una forma abreviada de escribir `1000000000`, o mil millones.
Cuando dividimos una `Series` por un número, se dividen todos los elementos de la `Series`. De ahora en adelante, expresaremos las estimaciones de población en términos de miles de millones.

Podemos usar `tail` para ver los últimos elementos de la `Series`:


In [11]:
census.tail()

La columna izquierda es el *índice* de la `Series`; en este ejemplo, contiene las fechas.
La columna derecha contiene los *valores*, que son estimaciones de población.
En 2016, la población mundial fue de aproximadamente 7.3 mil millones.

Aquí están las estimaciones del Departamento de Asuntos Económicos y Sociales de las Naciones Unidas (U.N. DESA):


In [12]:
un = table2.un / 1e9
un.tail()

La estimación más reciente que tenemos de la ONU es para 2015, por lo que el valor de 2016 es `NaN`.

Ahora podemos graficar las estimaciones así:


In [13]:
def plot_estimates():
    census.plot(style=':', label='US Census')
    un.plot(style='--', label='UN DESA')
    decorar(xlabel='Year', 
             ylabel='World population (billions)') 

El argumento con nombre `style=':'` especifica una línea punteada; `style='--'` especifica una línea discontinua. El argumento `label` proporciona el texto que aparece en la leyenda.

Y así es como se ve.


In [14]:
plot_estimates()
decorar(title='World population estimates')

Las líneas se superponen casi por completo, pero las estimaciones más recientes divergen ligeramente. En la siguiente sección, cuantificaremos estas diferencias.


## Errores Absolutos y Relativos

Las estimaciones de la población mundial del Censo de EE. UU. y de la ONU (DESA) difieren ligeramente.
Una forma de caracterizar esta diferencia es el *error absoluto*, que es el valor absoluto de la diferencia entre las estimaciones.

Para calcular los errores absolutos, podemos importar `abs` de NumPy:

In [15]:
from numpy import abs

Y la usamos 

In [16]:
abs_error = abs(un - census)
abs_error.tail()

Cuando restas dos objetos `Series`, el resultado es una nueva `Series`.
Como una de las estimaciones para 2016 es `NaN`, el resultado para 2016 también es `NaN`.

Para resumir los resultados, podemos calcular el *error absoluto medio*.

In [17]:
from numpy import mean

mean(abs_error)

En promedio, las estimaciones difieren en alrededor de 0.029 mil millones. Pero también podemos usar `max` para calcular el error absoluto máximo.

In [18]:
from numpy import max

max(abs_error)

En el peor de los casos, difieren en alrededor de 0.1 mil millones.

Ahora bien, 0.1 mil millones es mucha gente, por lo que podría sonar como una discrepancia seria.
Pero contar a todas las personas en el mundo es difícil, y no deberíamos esperar que las estimaciones sean exactas.

Otra forma de cuantificar la magnitud de la diferencia es el *error relativo*, que es el tamaño del error dividido entre las propias estimaciones.

In [19]:
rel_error = 100 * abs_error / census
rel_error.tail()

Multiplicamos por 100 para que podamos interpretar los resultados como un porcentaje.
En 2015, la diferencia entre las estimaciones es de aproximadamente 1.4%, y esa resulta ser la máxima.

Nuevamente, podemos resumir los resultados calculando la media.


In [20]:
mean(rel_error)

El error relativo medio es de aproximadamente 0.6%. Así que no está mal.

Podrían preguntarse por qué dividir entre `census` en lugar de `un`. En general, si crees que una estimación es mejor que la otra, colocas la mejor en el denominador. En este caso, no sé cuál es mejor, así que puse la más pequeña en el denominador, lo que hace que los errores calculados sean un poco más grandes.

## Modelado del Crecimiento Poblacional

Supongamos que queremos predecir el crecimiento de la población mundial durante los próximos 50 o 100 años. Podemos hacerlo desarrollando un modelo que describa cómo crecen las poblaciones, ajustando el modelo a los datos que tenemos hasta ahora y luego usando el modelo para generar predicciones. En las próximas secciones veremos este proceso comenzando con modelos simples y mejorándolos gradualmente.

Aunque en las estimaciones graficadas se aprecia cierta curvatura, parece que el crecimiento de la población mundial ha sido casi lineal desde 1960 aproximadamente.
Así que comenzaremos con un modelo de crecimiento constante. Para ajustar el modelo a los datos, calcularemos el crecimiento anual promedio de 1950 a 2016. Dado que los datos de la ONU y del Censo son muy cercanos, usaremos los del Censo. 

Podemos seleccionar un valor de un objeto `Series` usando el operador de corchetes:

In [21]:
census[1950]

Así podemos obtener el crecimiento total durante el intervalo de esta manera:

In [22]:
total_growth = census[2016] - census[1950]

En este ejemplo, las etiquetas 2016 y 1950 hacen parte de los datos, por lo que sería mejor no incluirlas directamente en el programa. Colocar valores como estos en el programa se llama *hard coding*; se considera una mala práctica porque si los datos cambian en el futuro, tendríamos que cambiar el programa,

Sería mejor obtener las etiquetas desde el objeto `Series`. Podemos hacerlo seleccionando el índice de `census` y luego el primer elemento.


In [23]:
t_0 = census.index[0]
t_0

Entonces, `t_0` es la etiqueta del primer elemento, que es 1950. Podemos obtener la etiqueta del último elemento de esta manera:

In [24]:
t_end = census.index[-1]
t_end

El valor `-1` indica el último elemento; `-2` indica el penúltimo, y así sucesivamente.

La diferencia entre `t_0` y `t_end` es el tiempo transcurrido entre ellos.

In [25]:
elapsed_time = t_end - t_0
elapsed_time

Ahora podemos usar `t_0` y `t_end` para seleccionar la población al inicio y al final del intervalo.

In [26]:
p_0 = census[t_0]
p_end = census[t_end]

Y calcular el crecimiento total durante el intervalo.

In [27]:
total_growth = p_end - p_0
total_growth

Finalmente, calculamos el promedio anual de crecimiento.

In [28]:
annual_growth = total_growth / elapsed_time
annual_growth

De 1950 a 2016, la población mundial creció en promedio alrededor de 0.07 mil millones de personas por año. 

El siguiente paso es usar esta estimación para simular el crecimiento poblacional.

## Simulación del crecimiento de la población

Nuestra simulación comenzará en 1950 con población inicial `p_0` y un crecimiento anual constante. Para almacenar los resultados, usaremos un objeto `TimeSeries`:


In [29]:
results = TimeSeries()

Podemos establecer el primer valor en la nueva `TimeSeries` de esta forma.


In [30]:
results[t_0] = p_0

Así se ve hasta ahora.


In [31]:
print(results)

Ahora establecemos el resto de los valores simulando el crecimiento anual:


In [32]:
for t in range(t_0, t_end):
    results[t+1] = results[t] + annual_growth

Los valores de `t` van desde `t_0` hasta `t_end`, incluyendo el primero pero no el último.

Dentro del bucle, calculamos la población para el año siguiente sumando la población del año actual y `annual_growth`. La última vez que pasa por el bucle, el valor de `t` es 2015, por lo que la última etiqueta en `results` es 2016.

Aquí están los resultados, comparados con las estimaciones.

In [33]:
results.plot(color='gray', label='model')
plot_estimates()
decorar(title='Constant growth model')

De 1950 a 1990, el modelo no ajusta los datos particularmente bien, pero después de eso es bastante bueno.


## Resumen

Este capítulo es un primer paso hacia el modelado de cambios demográficos utilizando datos reales. Cargamos datos de población, construimos un modelo simple de crecimiento constante y comparamos resultados con estimaciones. Según este modelo, la población mundial seguiría creciendo al mismo ritmo para siempre, lo cual no parece razonable.

En el siguiente capítulo consideraremos otros modelos que podrían ajustar mejor los datos y producir predicciones más creíbles.


Aquí está el código de este capítulo reunido en un solo lugar.


In [34]:
t_0 = census.index[0]
t_end = census.index[-1]
elapsed_time = t_end - t_0

p_0 = census[t_0]
p_end = census[t_end]

total_growth = p_end - p_0
annual_growth = total_growth / elapsed_time

results = TimeSeries()
results[t_0] = p_0

for t in range(t_0, t_end):
    results[t+1] = results[t] + annual_growth

In [35]:
results.plot(color='gray', label='model')
plot_estimates()
decorar(title='Constant growth model')

## Ejercicios

### Ejercicio 1

Intenta ajustar el modelo utilizando datos desde 1970 hasta la fecha y verifica si eso da un mejor resultado.

Sugerencias:

1. Define `t_1` como 1970 y `p_1` como la población en 1970 para estimar el crecimiento anual, pero usa `t_0` y `p_0` para ejecutar la simulación.

2. Quizá quieras añadir una constante al valor inicial para que coincida mejor con los datos.


In [ ]:
# La solución va aqui